In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, BitsAndBytesConfig
from datasets import load_dataset

In [ ]:
!pip install -q transformers torch bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.9 MB/s eta 0:00:00


## Phase 1: Model Setup & Bayesian Approximation

* Initialize the Student Model: Load a lightweight base model, such as Qwen-2.5-3B, to serve as your student.


* Enable Monte Carlo Dropout: Apply MC Dropout to the student model to approximate a Bayesian Neural Network. This ensures that estimating uncertainty remains computationally feasible across thousands of candidate trajectories.

In [ ]:
model_name = "Qwen/Qwen2.5-3B"

config = AutoConfig.from_pretrained(model_name)
config.attention_dropout = 0.1

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load the tokenizer and model with our custom config
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model... this may take a minute.")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    quantization_config=bnb_config,
    device_map="auto"
)

model.train()

torch.set_grad_enabled(False)

print("Model successfully loaded in training mode (MC Dropout active) with gradients disabled.")

Loading model... this may take a minute.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model successfully loaded in training mode (MC Dropout active) with gradients disabled.


In [ ]:
# Drop Out Verification

dummy_text = "To solve for x in the equation 2x + 5 = 15, the first step is"
inputs = tokenizer(dummy_text, return_tensors="pt").to(model.device)

print("Running forward pass 1...")
# Run the first pass and grab the raw logits (unnormalized probabilities)
outputs1 = model(**inputs)
logits1 = outputs1.logits

print("Running forward pass 2...")
# Run the second pass on the exact same input
outputs2 = model(**inputs)
logits2 = outputs2.logits

# Compare the two output tensors
# torch.allclose checks if all elements are within a tiny tolerance of each other
are_identical = torch.allclose(logits1, logits2, atol=1e-6)

# Calculate the actual numerical difference across the entire tensor
total_difference = (logits1 - logits2).abs().sum().item()

print("-" * 40)
print(f"Are the tensors completely identical? {are_identical}")
print(f"Total absolute variance between passes: {total_difference:.4f}")

if not are_identical and total_difference > 0:
    print("\n✅ SUCCESS: MC Dropout is active! The model is non-deterministic.")
else:
    print("\n❌ FAILURE: The model is deterministic. Dropout is NOT active.")

Running forward pass 1...
Running forward pass 2...
----------------------------------------
Are the tensors completely identical? False
Total absolute variance between passes: 3506176.0000

✅ SUCCESS: MC Dropout is active! The model is non-deterministic.


In [ ]:
print(logits1.shape)
print(logits2.shape)

torch.Size([1, 22, 151936])
torch.Size([1, 22, 151936])


## Phase 2: Uncertainty Extraction

* Generate Ensembles: For every candidate reasoning trajectory generated by a teacher model, perform multiple stochastic forward passes using teacher forcing to create an ensemble of predictions.

* Calculate Total and Aleatoric Uncertainty: From these approximate posteriors, compute both the predictive entropy (the model's total uncertainty) and the expected entropy (the aleatoric uncertainty, representing inherent noise or variable word choices).

* Isolate Epistemic Uncertainty: Subtract the expected entropy from the predictive entropy to find the Mutual Information. This isolates the epistemic uncertainty, mathematically pinpointing the model's actual lack of mathematical knowledge.

In [2]:
# Load the safe training dataset (ensuring no data leakage for MATH-500)
dataset = load_dataset("rasbt/math_full_minus_math500", split="train")

# Extract just the question strings to feed to your teacher
questions_for_teacher = dataset["problem"]

# Print the length to verify the scale of your candidate pool
total_questions = len(questions_for_teacher)
print(f"Total safe training questions loaded: {total_questions}")

# Print a sample to verify the formatting
print("\n--- Sample Question 0 ---")
print(questions_for_teacher[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.67M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12000 [00:00<?, ? examples/s]

Total safe training questions loaded: 12000

--- Sample Question 0 ---
Let \[f(x) = \left\{
\begin{array}{cl} ax+3, &\text{ if }x>2, \\
x-5 &\text{ if } -2 \le x \le 2, \\
2x-b &\text{ if } x <-2.
\end{array}
\right.\]Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper).


## Phase 3: Trajectory Scoring & Data Selection

* Compute Utility Scores: Calculate a top-m Mutual Information score for each trajectory, completely replacing the standard top-m entropy formulation.


* Filter Candidates: Normalize this new utility score and integrate it directly into your existing chi-squared-regularized closed-form selection rule. This systematic selection creates your curated offline dataset.

## Phase 4: Distillation

* Supervised Fine-Tuning (SFT): Distill the selected, high-value reasoning trajectories into your student model using behavior cloning. Because the data was selected based on epistemic uncertainty, the model is forced to resolve the specific mathematical reasoning knowledge it actually lacks

## Phase 5: Benchmarking & Evaluation

* Test Downstream Performance: Evaluate the fine-tuned student model's reasoning capabilities on standard benchmark datasets, such as MATH-500.


* Compare Against Baselines: Compare your model's sample efficiency and mathematical reasoning performance against a baseline model that was curated using the standard point-estimate entropy method.